## Importações

In [1]:
import pandas as pd
import numpy as np
import zipfile
from tqdm import trange
import matplotlib.pyplot as plt
from collections import namedtuple, deque
import random

## Tratamento de dados

In [2]:
path = "/content/archive.zip"
zip_obj = zipfile.ZipFile(path, 'r')
zip_obj.extractall()
zip_obj.close()

In [3]:
data = pd.read_csv("/content/AcoesBruto.csv", encoding="latin-1", on_bad_lines='skip', sep=";")
data.head()

<ipython-input-3-d24772f5ea25>:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("/content/AcoesBruto.csv", encoding="latin-1", on_bad_lines='skip', sep=";")


,price.open,price.high,price.low,price.close,volume,price.adjusted,ref.date,ticker,ret.adjusted.prices,ret.closing.prices,Empresa,Setor,Subsetor,Tipo,Classificação
0,"14,94","15,16","14,7","14,89",94500.0,"14,74",2018-01-02,AALR3.SA,NaN,NaN,ALLIAR,Saúde,Análises e Diagnósticos,Ordinaria,Small Cap
1,"14,89","15,32","14,79","14,96",584700.0,"14,81",2018-01-03,AALR3.SA,"0,00470117769938838","0,0047011417058429",ALLIAR,Saúde,Análises e Diagnósticos,Ordinaria,Small Cap
2,15,"15,16","14,81","15,09",290100.0,"14,94",2018-01-04,AALR3.SA,"0,00868974180746784","0,00868983957219238",ALLIAR,Saúde,Análises e Diagnósticos,Ordinaria,Small Cap
3,"15,05","15,17","14,95","15,05",131500.0,"14,9",2018-01-05,AALR3.SA,"-0,00265069648310279","-0,00265076209410198",ALLIAR,Saúde,Análises e Diagnósticos,Ordinaria,Small Cap
4,"15,05","15,1","14,62","14,79",244500.0,"14,64",2018-01-08,AALR3.SA,"-0,017275687925577","-0,0172757475083057",ALLIAR,Saúde,Análises e Diagnósticos,Ordinaria,Small Cap


In [4]:
data.dropna(inplace=True)
drop = ['volume','ref.date', 'ticker', 'ret.adjusted.prices',
       'ret.closing.prices', 'Empresa', 'Setor', 'Subsetor', 'Tipo',
       'Classificação']
data.drop(drop, axis=1, inplace=True)
data.drop_duplicates(inplace=True)

## Replay Memory

In [5]:
data = data.apply(lambda x: x.str.replace(',', '.', regex=True) if x.dtype == "object" else x)
data = np.array(data.astype(float), dtype="float32")
env = data[:20000]

In [6]:
# Definição da memória de replay
class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=int(capacity))

    def push(self, state, action, next_state, reward):
        """Salva uma transição na memória"""
        self.memory.append(Transition(state, action, next_state, reward))

    def sample(self, batch_size):
        """Amostra um lote de transições aleatoriamente"""
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [7]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

In [8]:
BUFFER_SIZE = int(1e5)  # Tamanho do buffer de replay
memory = ReplayMemory(BUFFER_SIZE) # Inicializa a memória de replay com a capacidade especificada

## Modelo

In [9]:
# Definição do agente IaTrader
class IaTrader:
    def __init__(self, num_actions, num_states, BUFFER_SIZE=1e5, BATCH_SIZE=128):
        self.num_actions = num_actions
        self.num_states = num_states
        self.q_table = np.zeros((num_states, num_actions))
        self.target_q_table = np.zeros_like(self.q_table)
        self.gamma = 0.99
        self.alpha = 0.1
        self.start = 0.9
        self.final_epsilon = 0.01
        self.epsilon_decay = 1000
        self.steps_done = 0
        self.memory = ReplayMemory(BUFFER_SIZE)
        self.BATCH_SIZE = BATCH_SIZE

    def get_action(self, state):
        self.epsilon = self.final_epsilon + (self.start - self.final_epsilon) * np.exp(-1.0 * self.steps_done / self.epsilon_decay)
        self.steps_done += 1

        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        else:
            return np.argmax(self.q_table[state])

    # def learn(self, state, action, reward, next_state):
    #     if next_state is not None and next_state < self.num_states:
    #         target = reward + self.gamma * np.max(self.q_table[next_state])
    #     else:
    #         target = reward  # Se next_state for None, não há próximo estado para considerar
    #     self.q_table[state, action] = (1 - self.alpha) * self.q_table[state, action] + self.alpha * target
    def learn(self, state, action, reward, next_state):
      if next_state < self.num_states:
          target = (reward + self.gamma * np.max(self.q_table[next_state]))
          self.q_table[state, action] = (1 - self.alpha) * self.q_table[state, action] + self.alpha * target

    def update(self):
        if len(self.memory) < self.BATCH_SIZE:
            return

        transitions = self.memory.sample(self.BATCH_SIZE)
        batch = Transition(*zip(*transitions))
        non_final_mask = np.array([s is not None for s in batch.next_state], dtype=bool)
        non_final_next_states = np.array([s for s in batch.next_state if s is not None])

        state_batch = np.array(batch.state)
        action_batch = np.array(batch.action)
        reward_batch = np.array(batch.reward)

        state_action_values = self.q_table[state_batch, action_batch]
        next_state_values = np.zeros(self.BATCH_SIZE)
        if len(non_final_next_states) > 0:
            next_state_values[non_final_mask - 1] = np.max(self.target_q_table[non_final_next_states], axis=1)
        expected_state_action_values = reward_batch + (self.gamma * next_state_values)

        for i in range(self.BATCH_SIZE):
            self.q_table[state_batch[i], action_batch[i]] += self.alpha * (expected_state_action_values[i] - self.q_table[state_batch[i], action_batch[i]])


In [10]:
num_actions = 3 # 0: manter, 1: comprar, 2: vender
num_states = env.shape[0]
agent = IaTrader(num_actions, num_states)
episodes = 100
total_profit = []

In [ ]:
for episode in range(1, episodes + 1):
    print(f"Episódio {episode}/{episodes}")

    profit_episode = 0
    inventory = []

    state = 0  # Estado inicial

    for s in trange(len(env)):
        action = agent.get_action(state)
        reward = 0

        if action == 1:  # Comprar ação
            inventory.append(env[state, 0]) # compra -> 0

        elif action == 2 and len(inventory) > 0:  # Vender ação
            reward = env[s, 3] - inventory.pop(0) # venda -> 3
            profit_episode += reward

        next_state = s + 1  # Supondo que o próximo estado seja o próximo passo no tempo

        done = (s == len(env) - 1)

        # Armazenar experiência na memória
        agent.memory.push(state, action, next_state, reward)
        agent.learn(state, action, reward, next_state)

        # Atualizar estado atual para próximo estado
        state = next_state

    agent.update()
    total_profit.append(profit_episode)

    # Resumo do fim do episódio
    print("############################################################")
    print(f"Lucro total no episódio {episode}: R$ {profit_episode:.2f}")
    print("############################################################")

print(f"Lucro total no treinamento {episode}: R$ {sum(total_profit):.2f}")
# Plotando o lucro total por episódio
plt.plot(total_profit)
plt.xlabel('Episódios')
plt.ylabel('Lucro Total por Episódio')
plt.title('Lucro Total ao Longo dos Episódios')
plt.show()